# GraphCodeBERT — Multi-Language Code Clone Detection

This notebook fine-tunes **GraphCodeBERT** (`microsoft/graphcodebert-base`) to classify pairs of code snippets (Python, Java, C) as semantic clones or not.

GraphCodeBERT extends RoBERTa by incorporating **Data Flow Graphs (DFG)** extracted from source code via tree-sitter AST parsing, enabling the model to reason about variable dependencies and program semantics beyond surface-level token patterns.

---

**Table of Contents**
1. [Setup & Imports](#1)
2. [Configuration](#2)
3. [Dataset Preparation & Exploration](#3)
4. [Parser & Data Flow Graph Setup](#4)
5. [Feature Engineering & Data Loading](#5)
6. [Model Definition](#6)
7. [Training](#7)
8. [Evaluation](#8)
9. [Test & Predictions](#9)
10. [Results Summary Dashboard](#10)

---
## 1. Setup & Imports <a id='1'></a>

Install required packages and import all dependencies. We also set the random seed and detect whether a GPU is available.

In [ ]:
%matplotlib inline

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "seaborn", "networkx", "scikit-learn", "matplotlib", "tqdm",
    "ipywidgets",
])

import os
import sys
import json
import random
import logging
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset, SequentialSampler, RandomSampler

from transformers import (
    RobertaConfig,
    RobertaForSequenceClassification,
    RobertaTokenizer,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from tqdm.auto import tqdm

from sklearn.metrics import (
    recall_score, precision_score, f1_score, accuracy_score,
    confusion_matrix, classification_report,
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print("networkx not installed — DFG graph drawing will be skipped.")

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    datefmt='%m/%d/%Y %H:%M:%S',
    level=logging.WARNING,
)
logger = logging.getLogger(__name__)

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu  = torch.cuda.device_count()

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')
print(f'Number of GPUs  : {n_gpu}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')

---
## 2. Configuration <a id='2'></a>

All hyperparameters and paths are collected in a single `Args` class, replacing the CLI `argparse` interface used in `run.py`.  
Edit the values here to change the training behaviour.

In [ ]:
class Args:
    # ── Model ──────────────────────────────────────────────────────────────
    model_name_or_path = 'microsoft/graphcodebert-base'
    tokenizer_name     = 'microsoft/graphcodebert-base'
    config_name        = 'microsoft/graphcodebert-base'

    # ── Data paths (relative to the clonedetection/ directory) ─────────────
    train_data_file = 'dataset_parent_03-31/train.txt'
    eval_data_file  = 'dataset_parent_03-31/valid.txt'
    test_data_file  = 'dataset_parent_03-31/test.txt'
    output_dir      = 'saved_models/multi_model_clean_8ksample-unchanged'

    # ── Sequence lengths ────────────────────────────────────────────────────
    code_length      = 256
    data_flow_length = 64

    # ── Training hyper-params ───────────────────────────────────────────────
    # RTX 3060 Ti (8 GB VRAM) — batch 4 keeps memory safe with code_length=256
    train_batch_size             = 4
    eval_batch_size              = 8
    learning_rate                = 2e-5
    weight_decay                 = 0.0
    adam_epsilon                 = 1e-8
    max_grad_norm                = 1.0
    epochs                       = 5
    gradient_accumulation_steps  = 1
    evaluate_during_training     = True

    # ── Misc ────────────────────────────────────────────────────────────────
    seed     = 123456
    do_train = True
    do_eval  = True
    do_test  = True

    # ── Runtime (set automatically) ─────────────────────────────────────────
    device = device
    n_gpu  = n_gpu

args = Args()

# Derived training-schedule values (populated later inside the training cell)
args.max_steps    = -1
args.save_steps   = -1
args.warmup_steps = -1

os.makedirs(args.output_dir, exist_ok=True)

# ── Reproducibility ──────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

set_seed(args.seed)

print('Configuration:')
for k, v in vars(Args).items():
    if not k.startswith('_') and not callable(v):
        print(f'  {k:35s} = {v}')

---
## 3. Dataset Preparation & Exploration <a id='3'></a>

The dataset contains **multi-language** code clone pairs (Python, Java, C).  
- `data.jsonl` — one JSON object per line with `idx` (function ID) and `func` (source code).
- `train.txt` / `valid.txt` / `test.txt` — whitespace-separated rows: `url1  url2  label  [lang]` where label `1` = clone, `0` = non-clone, and `lang` is one of `python`, `java`, or `c`.

We verify the dataset directory, then explore its contents with descriptive statistics and visualisations.

In [ ]:
# ── 3.1  Verify dataset directory exists ───────────────────────────────────────
DATASET_DIR = 'dataset_parent_03-31'

assert os.path.isdir(DATASET_DIR), (
    f'Dataset directory "{DATASET_DIR}" not found.\n'
    f'Place your data.jsonl, train.txt, valid.txt, and test.txt inside this folder.'
)

print(f'Dataset directory: {DATASET_DIR}')
for fname in ['data.jsonl', 'train.txt', 'valid.txt', 'test.txt']:
    path = os.path.join(DATASET_DIR, fname)
    exists = os.path.exists(path)
    print(f'  {path:45s}  {"✓" if exists else "✗ MISSING"}')

In [ ]:
# ── 3.2  Load data.jsonl and display sample entries ───────────────────────────
url_to_code = {}
with open(os.path.join(DATASET_DIR, 'data.jsonl'), encoding='utf-8') as f:
    for line in f:
        js = json.loads(line.strip())
        url_to_code[js['idx']] = js['func']

print(f'Total functions in data.jsonl: {len(url_to_code):,}')
print('\n--- Sample functions ---')
for i, (idx, func) in enumerate(list(url_to_code.items())[:3]):
    print(f'\n[Function {idx}]')
    print(func[:400] + ('…' if len(func) > 400 else ''))
    print('─' * 70)

In [ ]:
# ── 3.3  Load split index files and compute label distributions ───────────────
def load_split(path):
    """Return list of (url1, url2, label, lang) tuples from an index file."""
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            url1, url2, label = parts[0], parts[1], parts[2]
            lang = parts[3] if len(parts) >= 4 else 'python'
            if url1 in url_to_code and url2 in url_to_code:
                pairs.append((url1, url2, int(label), lang))
    return pairs

train_pairs = load_split(args.train_data_file)
valid_pairs = load_split(args.eval_data_file)
test_pairs  = load_split(args.test_data_file)

for name, pairs in [('Train', train_pairs), ('Valid', valid_pairs), ('Test', test_pairs)]:
    labels = [p[2] for p in pairs]
    langs  = Counter(p[3] for p in pairs)
    n_clone     = sum(labels)
    n_non_clone = len(labels) - n_clone
    print(f'{name:6s}: {len(pairs):>8,} pairs  |  clones={n_clone:>7,}  non-clones={n_non_clone:>7,}')
    print(f'        Languages: {dict(langs)}')

In [ ]:
# ── 3.4  Visualisations ──────────────────────────────────────────────────────
splits      = ['Train', 'Valid', 'Test']
all_pairs   = [train_pairs, valid_pairs, test_pairs]
clone_counts     = [sum(p[2] for p in ps) for ps in all_pairs]
non_clone_counts = [len(ps) - c for ps, c in zip(all_pairs, clone_counts)]
total_counts     = [len(ps) for ps in all_pairs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Bar chart: label distribution per split ──────────────────────────────────
ax = axes[0]
x = np.arange(len(splits))
w = 0.35
bars1 = ax.bar(x - w/2, clone_counts,     w, label='Clone (1)',     color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + w/2, non_clone_counts, w, label='Non-Clone (0)', color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(splits, fontsize=12)
ax.set_ylabel('Number of pairs', fontsize=11)
ax.set_title('Label Distribution per Split', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(total_counts)*0.005,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

# ── Pie chart: dataset split sizes ───────────────────────────────────────────
ax2 = axes[1]
colors = ['steelblue', 'mediumseagreen', 'darkorange']
wedges, texts, autotexts = ax2.pie(
    total_counts, labels=splits, autopct='%1.1f%%',
    colors=colors, startangle=90,
    textprops={'fontsize': 11},
)
for at in autotexts:
    at.set_fontsize(10)
ax2.set_title('Dataset Split Sizes', fontsize=13, fontweight='bold')
ax2.legend(
    [f'{s}: {n:,}' for s, n in zip(splits, total_counts)],
    loc='lower left', fontsize=10,
)

plt.tight_layout()
plt.show()

---
## 4. Parser & Data Flow Graph Setup <a id='4'></a>

GraphCodeBERT augments code tokens with **Data Flow Graph (DFG) nodes** — variable occurrences whose edges encode data-flow dependencies (e.g., `x` at line 5 depends on `x` defined at line 2).

The **tree-sitter** library parses source code into an AST, and `DFG.py` extracts these edges.  
We first build the shared parser library (`parser/my-languages.so`) from the tree-sitter grammars for Python, Java, and C, then load the parsers.

In [ ]:
# ── 4.1  Build tree-sitter shared library (if needed) ─────────────────────────
from tree_sitter import Language

SO_PATH = os.path.join('parser', 'my-languages.so')

if not os.path.exists(SO_PATH):
    print('Building parser/my-languages.so from tree-sitter grammars …')
    Language.build_library(
        SO_PATH,
        [
            os.path.join('parser', 'tree-sitter-python'),
            os.path.join('parser', 'tree-sitter-java'),
            os.path.join('parser', 'tree-sitter-c'),
        ],
    )
    print('Done.')
else:
    print(f'{SO_PATH} already exists — skipping build.')

In [ ]:
# ── 4.2  Load tree-sitter parsers ─────────────────────────────────────────────
sys.path.insert(0, os.path.abspath('.'))

from parser import (
    DFG_python, DFG_java, DFG_c,
    remove_comments_and_docstrings,
    tree_to_token_index,
    index_to_code_token,
    tree_to_variable_index,
)
from tree_sitter import Parser as TSParser

dfg_function = {
    'python': DFG_python,
    'java':   DFG_java,
    'c':      DFG_c,
}

parsers = {}
for lang, dfg_fn in dfg_function.items():
    LANGUAGE = Language(SO_PATH, lang)
    parser   = TSParser()
    parser.set_language(LANGUAGE)
    parsers[lang] = [parser, dfg_fn]

print('Parsers loaded for languages:', list(parsers.keys()))

In [1]:
# ── 4.3  extract_dataflow function (same logic as run.py) ─────────────────────
import signal, threading

def _extract_dataflow_inner(code, parser, lang):
    """Core DFG extraction (no timeout)."""
    try:
        code = remove_comments_and_docstrings(code, lang)
    except Exception:
        pass

    if lang == 'php':
        code = '<?php' + code + '?>'

    try:
        tree        = parser[0].parse(bytes(code, 'utf8'))
        root_node   = tree.root_node
        tokens_index = tree_to_token_index(root_node)
        code_lines  = code.split('\n')
        code_tokens = [index_to_code_token(x, code_lines) for x in tokens_index]

        index_to_code_map = {}
        for idx, (index, tok) in enumerate(zip(tokens_index, code_tokens)):
            index_to_code_map[index] = (idx, tok)

        try:
            DFG, _ = parser[1](root_node, index_to_code_map, {})
        except Exception:
            DFG = []

        DFG = sorted(DFG, key=lambda x: x[1])
        indexs = set()
        for d in DFG:
            if len(d[-1]) != 0:
                indexs.add(d[1])
            for x in d[-1]:
                indexs.add(x)
        dfg = [d for d in DFG if d[1] in indexs]
    except Exception:
        code_tokens, dfg = [], []

    return code_tokens, dfg


def extract_dataflow(code, parser, lang, timeout_sec=10):
    """Extract DFG with a timeout. Returns empty lists if extraction hangs."""
    result = [[], []]

    def target():
        result[0], result[1] = _extract_dataflow_inner(code, parser, lang)

    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join(timeout=timeout_sec)
    if t.is_alive():
        # Thread is still running — skip this snippet
        return [], []
    return result[0], result[1]

In [ ]:
# ── 4.4  Visualise DFG for a sample snippet per language ──────────────────────
# Search across all splits to find one sample per language
all_pairs = train_pairs + valid_pairs + test_pairs
shown_langs = set()

for url1, _, _, lang in all_pairs:
    if lang not in shown_langs and lang in parsers:
        shown_langs.add(lang)
        code = url_to_code[url1]
        code_tokens, dfg = extract_dataflow(code, parsers[lang], lang)

        print(f'{"═" * 70}')
        print(f'Language       : {lang}')
        print(f'Function index : {url1}')
        print(f'Code tokens    : {len(code_tokens)}')
        print(f'DFG edges      : {len(dfg)}')
        print(f'\n--- First 15 code tokens ---')
        print(code_tokens[:15])
        print(f'\n--- DFG edges (first 5) ---')
        for edge in dfg[:5]:
            print(f'  {edge}')
        print()

    if shown_langs == set(parsers.keys()):
        break

missing = set(parsers.keys()) - shown_langs
if missing:
    print(f'⚠ No samples found for: {missing}')

In [ ]:
# ── 4.5  Draw DFG with networkx for each language ─────────────────────────────
if HAS_NX:
    MAX_DFG_EDGES = 40          # use more edges so C isn't starved
    all_pairs_dfg = train_pairs + valid_pairs + test_pairs
    lang_samples = {}
    for url1, _, _, lang in all_pairs_dfg:
        if lang not in lang_samples and lang in parsers:
            code = url_to_code[url1]
            ct, dg = extract_dataflow(code, parsers[lang], lang)
            if len(dg) > 0:
                lang_samples[lang] = (ct, dg)
        if len(lang_samples) == len(parsers):
            break

    fig, axes = plt.subplots(1, len(lang_samples), figsize=(7 * len(lang_samples), 6))
    if len(lang_samples) == 1:
        axes = [axes]

    for ax, (lang, (ct, dg)) in zip(axes, lang_samples.items()):
        G = nx.DiGraph()
        edges_used = dg[:MAX_DFG_EDGES]

        # Determine the max token index referenced by any DFG edge so we
        # create enough token nodes for origin edges to land on.
        max_tok = max((e[1] for e in edges_used), default=0)
        tok_limit = min(max_tok + 1, len(ct))

        for i, tok in enumerate(ct[:tok_limit]):
            G.add_node(f'T{i}', label=tok[:8], node_type='token')

        # PASS 1 — add all DFG variable nodes first
        for edge in edges_used:
            var_name, src_idx = edge[0], edge[1]
            node_id = f'V{src_idx}:{var_name}'
            G.add_node(node_id, label=var_name[:8], node_type='var')
            if src_idx < tok_limit:
                G.add_edge(f'T{src_idx}', node_id, etype='origin')

        # PASS 2 — add flow edges (all targets guaranteed to exist)
        for edge in edges_used:
            var_name, src_idx = edge[0], edge[1]
            node_id = f'V{src_idx}:{var_name}'
            for src_var, d in zip(edge[3], edge[4]):
                dst_node = f'V{d}:{src_var}'
                if dst_node in G:
                    G.add_edge(node_id, dst_node, etype='flow')

        # Keep only DFG nodes + token nodes that connect to them
        dfg_nodes   = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'var'
                       and G.degree(n) > 0]
        token_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'token'
                       and G.degree(n) > 0]
        SG = G.subgraph(dfg_nodes + token_nodes)

        if len(SG.nodes) > 0:
            pos = nx.spring_layout(SG, seed=42, k=1.5)
            node_colors = [
                'lightblue' if SG.nodes[n].get('node_type') == 'token' else 'lightsalmon'
                for n in SG.nodes
            ]
            labels = {n: SG.nodes[n].get('label', n) for n in SG.nodes}
            nx.draw_networkx_nodes(SG, pos, node_color=node_colors, node_size=500, ax=ax)
            nx.draw_networkx_labels(SG, pos, labels=labels, font_size=6, ax=ax)
            nx.draw_networkx_edges(SG, pos, arrows=True, arrowsize=12, edge_color='grey', ax=ax)
        ax.set_title(f'DFG — {lang.capitalize()}', fontsize=13, fontweight='bold')
        ax.axis('off')

    token_patch = mpatches.Patch(color='lightblue',   label='Code token')
    var_patch   = mpatches.Patch(color='lightsalmon', label='DFG variable node')
    fig.legend(handles=[token_patch, var_patch], loc='lower center', ncol=2, fontsize=10)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()
else:
    print('Skipping DFG graph drawing (networkx not available).')

---
## 5. Feature Engineering & Data Loading <a id='5'></a>

Each code snippet is tokenised with the GraphCodeBERT tokeniser and padded to `code_length + data_flow_length` tokens.  DFG nodes are appended **after** the code tokens with position index `0` (a sentinel that distinguishes them from code tokens).  

A **graph-guided attention mask** is then built in `TextDataset.__getitem__` so that:
- Code tokens can attend to all other code tokens.
- DFG nodes only attend to the code tokens they originated from, and to adjacent DFG nodes.

In [ ]:
# ── 5.1  InputFeatures data class ─────────────────────────────────────────────
class InputFeatures:
    """Tokenised feature pair for a single clone-detection example."""

    def __init__(
        self,
        input_tokens_1, input_ids_1, position_idx_1, dfg_to_code_1, dfg_to_dfg_1,
        input_tokens_2, input_ids_2, position_idx_2, dfg_to_code_2, dfg_to_dfg_2,
        label, url1, url2,
    ):
        self.input_tokens_1 = input_tokens_1
        self.input_ids_1    = input_ids_1
        self.position_idx_1 = position_idx_1
        self.dfg_to_code_1  = dfg_to_code_1
        self.dfg_to_dfg_1   = dfg_to_dfg_1

        self.input_tokens_2 = input_tokens_2
        self.input_ids_2    = input_ids_2
        self.position_idx_2 = position_idx_2
        self.dfg_to_code_2  = dfg_to_code_2
        self.dfg_to_dfg_2   = dfg_to_dfg_2

        self.label = label
        self.url1  = url1
        self.url2  = url2

In [ ]:
# ── 5.2  convert_examples_to_features ────────────────────────────────────────
def convert_examples_to_features(item):
    """Tokenise one (url1, url2, label) pair and return an InputFeatures object."""
    url1, url2, label, lang, tokenizer, args, cache, url_to_code = item
    lang_parser = parsers[lang]

    for url in [url1, url2]:
        if url not in cache:
            func = url_to_code[url]

            # Extract data flow
            code_tokens, dfg = extract_dataflow(func, lang_parser, lang)
            code_tokens = [
                tokenizer.tokenize('@ ' + x)[1:] if idx != 0
                else tokenizer.tokenize(x)
                for idx, x in enumerate(code_tokens)
            ]
            ori2cur_pos = {-1: (0, 0)}
            for i in range(len(code_tokens)):
                ori2cur_pos[i] = (
                    ori2cur_pos[i - 1][1],
                    ori2cur_pos[i - 1][1] + len(code_tokens[i]),
                )
            code_tokens = [tok for sub in code_tokens for tok in sub]

            # Truncate
            max_code = args.code_length + args.data_flow_length - 3 - min(len(dfg), args.data_flow_length)
            code_tokens  = code_tokens[:max_code][:512 - 3]
            source_tokens = [tokenizer.cls_token] + code_tokens + [tokenizer.sep_token]
            source_ids    = tokenizer.convert_tokens_to_ids(source_tokens)
            position_idx  = [i + tokenizer.pad_token_id + 1 for i in range(len(source_tokens))]

            dfg = dfg[:args.code_length + args.data_flow_length - len(source_tokens)]
            source_tokens += [x[0] for x in dfg]
            position_idx  += [0] * len(dfg)
            source_ids    += [tokenizer.unk_token_id] * len(dfg)

            padding_length = args.code_length + args.data_flow_length - len(source_ids)
            position_idx   += [tokenizer.pad_token_id] * padding_length
            source_ids     += [tokenizer.pad_token_id] * padding_length

            # Reindex DFG edges
            reverse_index = {x[1]: idx for idx, x in enumerate(dfg)}
            dfg = [
                x[:-1] + ([reverse_index[i] for i in x[-1] if i in reverse_index],)
                for x in dfg
            ]
            dfg_to_dfg  = [x[-1] for x in dfg]
            dfg_to_code = [ori2cur_pos[x[1]] for x in dfg]
            length      = 1  # length of [CLS]
            dfg_to_code = [(a + length, b + length) for a, b in dfg_to_code]

            cache[url] = source_tokens, source_ids, position_idx, dfg_to_code, dfg_to_dfg

    t1, i1, p1, dc1, dd1 = cache[url1]
    t2, i2, p2, dc2, dd2 = cache[url2]
    return InputFeatures(t1, i1, p1, dc1, dd1, t2, i2, p2, dc2, dd2, label, url1, url2)

In [ ]:
# ── 5.3  TextDataset ──────────────────────────────────────────────────────────
class TextDataset(Dataset):
    """Loads a split index file and converts all pairs to InputFeatures."""

    def __init__(self, tokenizer, args, file_path='train'):
        self.examples = []
        self.args = args

        # Load code snippets from data.jsonl (same directory as the split file)
        data_dir = os.path.dirname(file_path)
        url_to_code_local = {}
        with open(os.path.join(data_dir, 'data.jsonl'), encoding='utf-8') as f:
            for line in f:
                js = json.loads(line.strip())
                url_to_code_local[js['idx']] = js['func']

        data  = []
        cache = {}
        with open(file_path, encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                url1, url2, label = parts[0], parts[1], parts[2]
                lang = parts[3] if len(parts) >= 4 else 'python'
                if url1 not in url_to_code_local or url2 not in url_to_code_local:
                    continue
                label = 1 if label != '0' else 0
                data.append((url1, url2, label, lang, tokenizer, args, cache, url_to_code_local))

        # Use only 10 % of validation data (speeds up mid-training checkpoints)
        if 'valid' in file_path:
            data = random.sample(data, int(len(data) * 0.1))

        self.examples = [
            convert_examples_to_features(x)
            for x in tqdm(data, desc=f'Loading {os.path.basename(file_path)}')
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, item):
        L = self.args.code_length + self.args.data_flow_length

        def build_attn_mask(pos_idx, input_ids, dfg_to_code, dfg_to_dfg):
            attn_mask  = np.zeros((L, L), dtype=bool)
            node_index = sum(i > 1 for i in pos_idx)
            max_length = sum(i != 1 for i in pos_idx)
            attn_mask[:node_index, :node_index] = True
            for idx, i in enumerate(input_ids):
                if i in [0, 2]:
                    attn_mask[idx, :max_length] = True
            for idx, (a, b) in enumerate(dfg_to_code):
                if a < node_index and b < node_index:
                    attn_mask[idx + node_index, a:b] = True
                    attn_mask[a:b, idx + node_index] = True
            for idx, nodes in enumerate(dfg_to_dfg):
                for a in nodes:
                    if a + node_index < len(pos_idx):
                        attn_mask[idx + node_index, a + node_index] = True
            return attn_mask

        ex = self.examples[item]
        attn_mask_1 = build_attn_mask(ex.position_idx_1, ex.input_ids_1, ex.dfg_to_code_1, ex.dfg_to_dfg_1)
        attn_mask_2 = build_attn_mask(ex.position_idx_2, ex.input_ids_2, ex.dfg_to_code_2, ex.dfg_to_dfg_2)

        return (
            torch.tensor(ex.input_ids_1),
            torch.tensor(ex.position_idx_1),
            torch.tensor(attn_mask_1),
            torch.tensor(ex.input_ids_2),
            torch.tensor(ex.position_idx_2),
            torch.tensor(attn_mask_2),
            torch.tensor(ex.label),
        )

In [ ]:
# ── 5.4  Show a sample processed feature per language ─────────────────────────
print('Loading tokenizer for feature demonstration …')
demo_tokenizer = RobertaTokenizer.from_pretrained(args.tokenizer_name)

all_pairs_demo = train_pairs + valid_pairs + test_pairs
demo_shown = set()
cache_demo = {}

for url1_demo, url2_demo, lbl_demo, lang_demo in all_pairs_demo:
    if lang_demo in demo_shown:
        continue
    demo_shown.add(lang_demo)

    feat = convert_examples_to_features(
        (url1_demo, url2_demo, lbl_demo, lang_demo, demo_tokenizer, args, cache_demo, url_to_code)
    )

    print(f'{"═" * 70}')
    print(f'Language        : {lang_demo}')
    print(f'Label           : {feat.label}')
    print(f'URL 1           : {feat.url1}')
    print(f'URL 2           : {feat.url2}')
    print(f'\nInput tokens 1  : {[t.replace(chr(288), "_") for t in feat.input_tokens_1[:20]]} …')
    print(f'Input IDs 1     : {feat.input_ids_1[:20]} …')
    print(f'Position idx 1  : {feat.position_idx_1[:20]} …')
    print(f'DFG→code 1      : {feat.dfg_to_code_1[:5]}')
    print(f'DFG→DFG  1      : {feat.dfg_to_dfg_1[:5]}')
    print()

    if demo_shown == set(parsers.keys()):
        break

In [ ]:
# ── 5.5  Visualise the graph-guided attention mask ────────────────────────────
L = args.code_length + args.data_flow_length   # 320

def build_attn_mask_demo(pos_idx, input_ids, dfg_to_code, dfg_to_dfg):
    attn_mask  = np.zeros((L, L), dtype=bool)
    node_index = sum(i > 1 for i in pos_idx)
    max_length = sum(i != 1 for i in pos_idx)
    attn_mask[:node_index, :node_index] = True
    for idx, i in enumerate(input_ids):
        if i in [0, 2]:
            attn_mask[idx, :max_length] = True
    for idx, (a, b) in enumerate(dfg_to_code):
        if a < node_index and b < node_index:
            attn_mask[idx + node_index, a:b] = True
            attn_mask[a:b, idx + node_index] = True
    for idx, nodes in enumerate(dfg_to_dfg):
        for a in nodes:
            if a + node_index < len(pos_idx):
                attn_mask[idx + node_index, a + node_index] = True
    return attn_mask

mask = build_attn_mask_demo(
    feat.position_idx_1, feat.input_ids_1, feat.dfg_to_code_1, feat.dfg_to_dfg_1
)

# Show only the non-padded part
non_pad = sum(i != 1 for i in feat.position_idx_1)
non_pad = max(non_pad, 1)
crop    = mask[:non_pad, :non_pad].astype(np.float32)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(crop, cmap='Blues', aspect='auto', interpolation='nearest')
ax.set_xlabel('Key position', fontsize=11)
ax.set_ylabel('Query position', fontsize=11)
ax.set_title('Graph-Guided Attention Mask (code snippet 1)', fontsize=13, fontweight='bold')

# Mark the DFG node boundary
node_index = sum(i > 1 for i in feat.position_idx_1)
if node_index < non_pad:
    ax.axvline(x=node_index - 0.5, color='red',  linewidth=1.5, linestyle='--', label='DFG boundary')
    ax.axhline(y=node_index - 0.5, color='red',  linewidth=1.5, linestyle='--')
    ax.legend(fontsize=10)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f'\nTotal positions : {L}')
print(f'Non-pad tokens  : {non_pad}  (code tokens up to position {node_index}, DFG nodes after)')
print(f'Attention density: {crop.mean():.3f}  ({crop.sum():.0f} / {crop.size} cells active)')

---
## 6. Model Definition <a id='6'></a>

The model wraps `RobertaForSequenceClassification` with a custom classification head that:
1. Computes averaged DFG-node embeddings from their corresponding code-token embeddings (via `einsum`).
2. Runs the combined (code + DFG) sequence through the RoBERTa encoder with the graph-guided attention mask.
3. Extracts the `[CLS]` token from **both** code snippets, concatenates them, and classifies via a 2-layer MLP.

In [ ]:
# ── 6.1  Model class (Cosine Similarity approach) ─────────────────────────────
# Numerical stability constant for DFG node embedding averaging
EPSILON = 1e-10


class Model(nn.Module):
    """GraphCodeBERT clone detection model using cosine similarity."""

    def __init__(self, encoder, config, tokenizer, args):
        super().__init__()
        self.encoder   = encoder
        self.config    = config
        self.tokenizer = tokenizer
        self.args      = args

    def forward(
        self,
        inputs_ids_1, position_idx_1, attn_mask_1,
        inputs_ids_2, position_idx_2, attn_mask_2,
        labels=None,
    ):
        bs, l = inputs_ids_1.size()
        inputs_ids   = torch.cat((inputs_ids_1.unsqueeze(1),   inputs_ids_2.unsqueeze(1)),   1).view(bs * 2, l)
        position_idx = torch.cat((position_idx_1.unsqueeze(1), position_idx_2.unsqueeze(1)), 1).view(bs * 2, l)
        attn_mask    = torch.cat((attn_mask_1.unsqueeze(1),    attn_mask_2.unsqueeze(1)),    1).view(bs * 2, l, l)

        # DFG node embedding: average over attended code tokens
        nodes_mask       = position_idx.eq(0)
        token_mask       = position_idx.ge(2)
        inputs_embeddings = self.encoder.roberta.embeddings.word_embeddings(inputs_ids)
        nodes_to_token_mask = nodes_mask[:, :, None] & token_mask[:, None, :] & attn_mask
        nodes_to_token_mask = nodes_to_token_mask / (nodes_to_token_mask.sum(-1) + EPSILON)[:, :, None]
        avg_embeddings   = torch.einsum('abc,acd->abd', nodes_to_token_mask, inputs_embeddings)
        inputs_embeddings = (
            inputs_embeddings * (~nodes_mask)[:, :, None]
            + avg_embeddings  * nodes_mask[:, :, None]
        )

        # Add position + token_type embeddings via the embeddings module
        embedding_output = self.encoder.roberta.embeddings(
            input_ids=None,
            position_ids=position_idx,
            token_type_ids=position_idx.eq(-1).long(),
            inputs_embeds=inputs_embeddings,
        )

        # Convert 3D boolean mask [batch, seq, seq] → 4D float mask [batch, 1, seq, seq]
        extended_attn_mask = attn_mask[:, None, :, :].to(dtype=embedding_output.dtype)
        extended_attn_mask = (1.0 - extended_attn_mask) * torch.finfo(embedding_output.dtype).min

        # Run through encoder layers directly
        outputs = self.encoder.roberta.encoder(
            embedding_output,
            attention_mask=extended_attn_mask,
        )[0]

        # Extract [CLS] embeddings and split back into pairs
        cls_embeddings = outputs[:, 0, :]                   # [bs*2, hidden]
        cls_embeddings = cls_embeddings.view(bs, 2, -1)     # [bs, 2, hidden]
        vec1 = cls_embeddings[:, 0, :]                      # [bs, hidden]
        vec2 = cls_embeddings[:, 1, :]                      # [bs, hidden]

        # Cosine similarity → mapped to [0, 1] for threshold compatibility
        cos_sim = F.cosine_similarity(vec1, vec2, dim=-1)   # [bs] in [-1, 1]
        scores  = (cos_sim + 1.0) / 2.0                    # [bs] in [0, 1]

        if labels is not None:
            # CosineEmbeddingLoss expects targets in {-1, +1}
            targets = labels * 2 - 1  # 0 → -1 (non-clone), 1 → +1 (clone)
            loss = nn.CosineEmbeddingLoss(margin=0.5)(vec1, vec2, targets.float())
            return loss, scores
        return scores

In [ ]:
# ── 6.2  Load pre-trained weights and print architecture summary ──────────────
print('Loading pre-trained GraphCodeBERT …')
config    = RobertaConfig.from_pretrained(args.config_name)
config.num_labels = 1
config.attn_implementation = "eager"   # required for custom graph-guided attention mask
tokenizer = RobertaTokenizer.from_pretrained(args.tokenizer_name)
encoder   = RobertaForSequenceClassification.from_pretrained(args.model_name_or_path, config=config)
model     = Model(encoder, config, tokenizer, args)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')

print('\nTop-level modules:')
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f'  {name:20s}  {n:>12,} params')

---
## 7. Training <a id='7'></a>

Trains the model and logs loss / learning rate / validation metrics at regular checkpoints.  
When `evaluate_during_training=True`, we evaluate every 10% of an epoch and save the checkpoint with the best F1.

Metrics collected:
- `train_losses` — smoothed loss per optimisation step
- `learning_rates` — LR schedule (linear warmup then decay)
- `eval_f1/precision/recall` — validation metrics at each checkpoint (with threshold sweep)

In [ ]:
# ── 7.1  evaluate() helper (used inside training loop) ───────────────────────
_eval_dataset_cache = None  # cache parsed validation features across calls

def evaluate(args, model, tokenizer, return_scores=False):
    """Run evaluation on the validation set. Sweeps thresholds 0.1–0.9 to find the best F1."""
    global _eval_dataset_cache
    if _eval_dataset_cache is None:
        _eval_dataset_cache = TextDataset(tokenizer, args, file_path=args.eval_data_file)
    eval_dataset = _eval_dataset_cache

    eval_sampler    = SequentialSampler(eval_dataset)
    eval_dataloader = DataLoader(
        eval_dataset, sampler=eval_sampler, batch_size=args.eval_batch_size, num_workers=0
    )

    model.eval()
    all_scores, all_labels = [], []
    for batch in tqdm(eval_dataloader, desc='Evaluating', leave=False):
        (
            inputs_ids_1, position_idx_1, attn_mask_1,
            inputs_ids_2, position_idx_2, attn_mask_2,
            labels,
        ) = [x.to(device) for x in batch]

        with torch.no_grad():
            _, scores = model(
                inputs_ids_1, position_idx_1, attn_mask_1,
                inputs_ids_2, position_idx_2, attn_mask_2,
                labels,
            )
        all_scores.append(scores.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_scores = np.concatenate(all_scores, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Sweep thresholds to maximise F1
    best_f1, best_threshold = 0, 0.5
    for thresh in np.arange(0.1, 0.9, 0.05):
        preds = (all_scores > thresh).astype(int)
        f = f1_score(all_labels, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_threshold = f, float(thresh)

    y_preds = (all_scores > best_threshold).astype(int)
    result = {
        'eval_recall':    float(recall_score(all_labels, y_preds,    zero_division=0)),
        'eval_precision': float(precision_score(all_labels, y_preds, zero_division=0)),
        'eval_f1':        float(f1_score(all_labels, y_preds,        zero_division=0)),
        'eval_threshold': best_threshold,
    }
    if return_scores:
        result['scores'] = all_scores
        result['labels'] = all_labels
    return result

In [ ]:
# ── 7.2  Training loop ────────────────────────────────────────────────────────
# Statistics containers
train_losses          = []   # (global_step, loss)
learning_rates        = []   # (global_step, lr)
eval_steps            = []   # global steps at which eval was run
eval_f1_scores        = []
eval_precision_scores = []
eval_recall_scores    = []

if args.do_train:
    set_seed(args.seed)

    # Load training data
    print('Loading training dataset …')
    train_dataset = TextDataset(tokenizer, args, file_path=args.train_data_file)
    train_sampler = RandomSampler(train_dataset)
    train_dataloader = DataLoader(
        train_dataset, sampler=train_sampler,
        batch_size=args.train_batch_size, num_workers=0,
    )

    # Training schedule
    args.max_steps    = args.epochs * len(train_dataloader)
    args.save_steps   = max(1, len(train_dataloader) // 5)
    args.warmup_steps = args.max_steps // 5

    # Optimizer
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': args.weight_decay},
        {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=args.learning_rate, eps=args.adam_epsilon)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=args.warmup_steps,
        num_training_steps=args.max_steps,
    )

    if args.n_gpu > 1:
        model = torch.nn.DataParallel(model)

    print(f'\n--- Training ---')
    print(f'  Examples             : {len(train_dataset):,}')
    print(f'  Epochs               : {args.epochs}')
    print(f'  Batch size           : {args.train_batch_size}')
    print(f'  Total steps          : {args.max_steps}')
    print(f'  Warmup steps         : {args.warmup_steps}')
    print(f'  Eval during training : {args.evaluate_during_training}')
    print(f'  Checkpoint every     : {args.save_steps} steps')

    global_step = 0
    tr_loss     = 0.0
    best_f1     = 0.0
    model.zero_grad()

    for epoch_idx in range(args.epochs):
        bar      = tqdm(train_dataloader, desc=f'Epoch {epoch_idx + 1}/{args.epochs}')
        tr_num   = 0
        epoch_loss = 0.0

        for step, batch in enumerate(bar):
            (
                inputs_ids_1, position_idx_1, attn_mask_1,
                inputs_ids_2, position_idx_2, attn_mask_2,
                labels,
            ) = [x.to(device) for x in batch]

            model.train()
            loss, _ = model(
                inputs_ids_1, position_idx_1, attn_mask_1,
                inputs_ids_2, position_idx_2, attn_mask_2,
                labels,
            )

            if args.n_gpu > 1:
                loss = loss.mean()
            if args.gradient_accumulation_steps > 1:
                loss = loss / args.gradient_accumulation_steps

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)

            tr_loss    += loss.item()
            epoch_loss += loss.item()
            tr_num     += 1
            avg_loss    = round(epoch_loss / tr_num, 5)
            bar.set_postfix(loss=avg_loss)

            if (step + 1) % args.gradient_accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1

                # Record training statistics
                current_lr = scheduler.get_last_lr()[0]
                train_losses.append((global_step, avg_loss))
                learning_rates.append((global_step, current_lr))

                # Checkpoint evaluation (only when evaluate_during_training is enabled)
                if args.evaluate_during_training and global_step % args.save_steps == 0:
                    results = evaluate(args, model, tokenizer)
                    eval_steps.append(global_step)
                    eval_f1_scores.append(results['eval_f1'])
                    eval_precision_scores.append(results['eval_precision'])
                    eval_recall_scores.append(results['eval_recall'])

                    print(
                        f'  step {global_step:5d} | '
                        f'loss={avg_loss:.4f} | '
                        f'F1={results["eval_f1"]:.4f} | '
                        f'P={results["eval_precision"]:.4f} | '
                        f'R={results["eval_recall"]:.4f} | '
                        f'thresh={results["eval_threshold"]:.2f}'
                    )

                    if results['eval_f1'] > best_f1:
                        best_f1  = results['eval_f1']
                        ckpt_dir = os.path.join(args.output_dir, 'checkpoint-best-f1')
                        os.makedirs(ckpt_dir, exist_ok=True)
                        model_to_save = model.module if hasattr(model, 'module') else model
                        torch.save(model_to_save.state_dict(), os.path.join(ckpt_dir, 'model.bin'))
                        print(f'  *** Best F1: {best_f1:.4f} — checkpoint saved ***')

    print(f'\nTraining complete. Best validation F1: {best_f1:.4f}')
else:
    print('Skipping training (do_train=False).')

In [ ]:
# ── 7.3  Training visualisations ──────────────────────────────────────────────
if len(train_losses) == 0:
    print('No training statistics to plot — run the training cell first.')
else:
    steps_loss, losses = zip(*train_losses)
    steps_lr,   lrs    = zip(*learning_rates)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── Training loss curve ───────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(steps_loss, losses, color='steelblue', linewidth=1.2, alpha=0.9)
    ax.set_xlabel('Optimisation Step', fontsize=11)
    ax.set_ylabel('Training Loss',     fontsize=11)
    ax.set_title('Training Loss Curve', fontsize=13, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.5)

    # ── Learning rate schedule ────────────────────────────────────────────────
    ax = axes[1]
    ax.plot(steps_lr, lrs, color='darkorange', linewidth=1.2)
    ax.set_xlabel('Optimisation Step', fontsize=11)
    ax.set_ylabel('Learning Rate',     fontsize=11)
    ax.set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.2e}'))

    # ── Validation metrics over checkpoints ───────────────────────────────────
    ax = axes[2]
    if len(eval_steps) > 0:
        ax.plot(eval_steps, eval_f1_scores,        marker='o', label='F1',        color='steelblue')
        ax.plot(eval_steps, eval_precision_scores,  marker='s', label='Precision', color='mediumseagreen')
        ax.plot(eval_steps, eval_recall_scores,     marker='^', label='Recall',    color='darkorange')
        ax.set_xlabel('Checkpoint Step', fontsize=11)
        ax.set_ylabel('Score',           fontsize=11)
        ax.set_title('Validation Metrics at Checkpoints', fontsize=13, fontweight='bold')
        ax.legend(fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.grid(True, linestyle='--', alpha=0.5)
    else:
        ax.text(0.5, 0.5, 'No checkpoint evaluations logged',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)

    plt.suptitle('Training Overview', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

---
## 8. Evaluation <a id='8'></a>

Loads the best checkpoint and evaluates on the validation set with a **threshold sweep** (0.1–0.9 in 0.05 steps) to find the F1-optimal decision boundary.

Visualisations: confusion matrix, ROC curve (AUC), and precision-recall curve (AP).

In [ ]:
# ── 8.1  Load best checkpoint ────────────────────────────────────────────────
CHECKPOINT_PATH = os.path.join(args.output_dir, 'checkpoint-best-f1', 'model.bin')

if args.do_eval:
    if not os.path.exists(CHECKPOINT_PATH):
        print(f'Checkpoint not found at {CHECKPOINT_PATH}. Run training first.')
    else:
        # Re-instantiate model cleanly to avoid DataParallel issues
        config_eval = RobertaConfig.from_pretrained(args.config_name)
        config_eval.num_labels = 1
        config_eval.attn_implementation = "eager"
        encoder_eval = RobertaForSequenceClassification.from_pretrained(
            args.model_name_or_path, config=config_eval
        )
        eval_model = Model(encoder_eval, config_eval, tokenizer, args)
        eval_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
        eval_model.to(device)
        eval_model.eval()
        print(f'Loaded checkpoint from {CHECKPOINT_PATH}')

        # Run evaluation (includes threshold sweep)
        results = evaluate(args, eval_model, tokenizer, return_scores=True)
        all_scores = results.pop('scores')
        all_labels = results.pop('labels')
        best_threshold = results['eval_threshold']
        y_preds    = (all_scores > best_threshold).astype(int)
        y_probs    = all_scores

        acc = accuracy_score(all_labels, y_preds)
        print(f'\n--- Validation Results ---')
        print(f'  Threshold : {best_threshold:.2f}')
        print(f'  Accuracy  : {acc:.4f}')
        print(f'  Precision : {results["eval_precision"]:.4f}')
        print(f'  Recall    : {results["eval_recall"]:.4f}')
        print(f'  F1        : {results["eval_f1"]:.4f}')
else:
    print('Skipping evaluation (do_eval=False).')

In [ ]:
# ── 8.2  Visualisations ──────────────────────────────────────────────────────
if args.do_eval and os.path.exists(CHECKPOINT_PATH):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── Confusion matrix ─────────────────────────────────────────────────────
    ax = axes[0]
    cm = confusion_matrix(all_labels, y_preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Non-Clone', 'Clone'],
        yticklabels=['Non-Clone', 'Clone'],
    )
    ax.set_xlabel('Predicted',  fontsize=11)
    ax.set_ylabel('Actual',     fontsize=11)
    ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')

    # ── ROC curve ─────────────────────────────────────────────────────────────
    ax = axes[1]
    fpr, tpr, _ = roc_curve(all_labels, y_probs)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {roc_auc:.4f}')
    ax.plot([0, 1], [0, 1], color='grey', linewidth=1, linestyle='--', label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate',  fontsize=11)
    ax.set_title('ROC Curve', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)

    # ── Precision-Recall curve ────────────────────────────────────────────────
    ax = axes[2]
    precision_vals, recall_vals, _ = precision_recall_curve(all_labels, y_probs)
    avg_prec = average_precision_score(all_labels, y_probs)
    ax.plot(recall_vals, precision_vals, color='darkorange', linewidth=2,
            label=f'AP = {avg_prec:.4f}')
    ax.set_xlabel('Recall',    fontsize=11)
    ax.set_ylabel('Precision', fontsize=11)
    ax.set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)

    plt.suptitle('Evaluation on Validation Set', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Classification report ─────────────────────────────────────────────────
    print('\n--- Classification Report ---')
    print(classification_report(all_labels, y_preds,
                                target_names=['Non-Clone', 'Clone'], digits=4))

---
## 9. Test & Predictions <a id='9'></a>

Run inference on the test split, write `predictions.txt` in the same format expected by `evaluator/evaluator.py` (`url1  url2  label`), and display sample predictions alongside ground-truth labels.

In [ ]:
# ── 9.1  Test inference ───────────────────────────────────────────────────────
if args.do_test:
    if not os.path.exists(CHECKPOINT_PATH):
        print(f'Checkpoint not found at {CHECKPOINT_PATH}. Run training first.')
    else:
        # Load best model (reuse from eval section if available)
        if 'eval_model' not in dir() or eval_model is None:
            config_test   = RobertaConfig.from_pretrained(args.config_name)
            config_test.num_labels = 1
            config_test.attn_implementation = "eager"
            encoder_test  = RobertaForSequenceClassification.from_pretrained(
                args.model_name_or_path, config=config_test
            )
            eval_model = Model(encoder_test, config_test, tokenizer, args)
            eval_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
            eval_model.to(device)

        # Use the threshold found during evaluation, or find it now
        if 'best_threshold' not in dir():
            eval_result = evaluate(args, eval_model, tokenizer)
            best_threshold = eval_result['eval_threshold']
            print(f'Threshold from eval sweep: {best_threshold:.2f}')

        eval_model.eval()

        test_dataset    = TextDataset(tokenizer, args, file_path=args.test_data_file)
        test_sampler    = SequentialSampler(test_dataset)
        test_dataloader = DataLoader(
            test_dataset, sampler=test_sampler, batch_size=args.eval_batch_size, num_workers=0
        )

        test_scores_all, test_labels_all = [], []
        for batch in tqdm(test_dataloader, desc='Test inference'):
            (
                inputs_ids_1, position_idx_1, attn_mask_1,
                inputs_ids_2, position_idx_2, attn_mask_2,
                labels,
            ) = [x.to(device) for x in batch]
            with torch.no_grad():
                _, scores = eval_model(
                    inputs_ids_1, position_idx_1, attn_mask_1,
                    inputs_ids_2, position_idx_2, attn_mask_2,
                    labels,
                )
            test_scores_all.append(scores.cpu().numpy())
            test_labels_all.append(labels.cpu().numpy())

        test_scores_all  = np.concatenate(test_scores_all,  axis=0)
        test_labels_all  = np.concatenate(test_labels_all,  axis=0)
        test_preds       = (test_scores_all > best_threshold).astype(int)

        # Write predictions.txt
        pred_path = os.path.join(args.output_dir, 'predictions.txt')
        with open(pred_path, 'w') as f:
            for example, pred in zip(test_dataset.examples, test_preds):
                f.write(f'{example.url1}\t{example.url2}\t{int(pred)}\n')
        print(f'Predictions written to {pred_path}  ({len(test_preds):,} examples)')

        # Test metrics
        test_f1  = f1_score(test_labels_all,        test_preds, zero_division=0)
        test_p   = precision_score(test_labels_all, test_preds, zero_division=0)
        test_r   = recall_score(test_labels_all,    test_preds, zero_division=0)
        test_acc = accuracy_score(test_labels_all,  test_preds)

        print(f'\n--- Test Results (threshold={best_threshold:.2f}) ---')
        print(f'  Accuracy  : {test_acc:.4f}')
        print(f'  Precision : {test_p:.4f}')
        print(f'  Recall    : {test_r:.4f}')
        print(f'  F1        : {test_f1:.4f}')
else:
    print('Skipping test (do_test=False).')

In [ ]:
# ── 9.2  Show sample predictions vs ground truth ──────────────────────────────
if args.do_test and os.path.exists(CHECKPOINT_PATH):
    N_SHOW = 10
    print(f'First {N_SHOW} test predictions:')
    print(f'{"URL1":>10}  {"URL2":>10}  {"Ground Truth":>13}  {"Prediction":>10}  {"Correct?":>8}')
    print('-' * 60)
    for i in range(min(N_SHOW, len(test_dataset.examples))):
        ex   = test_dataset.examples[i]
        pred = int(test_preds[i])
        gt   = int(test_labels_all[i])
        ok   = '✓' if pred == gt else '✗'
        print(f'{ex.url1:>10}  {ex.url2:>10}  {gt:>13}  {pred:>10}  {ok:>8}')

---
## 10. Results Summary Dashboard <a id='10'></a>

A consolidated view of all key metrics and comparative visualisations across train/eval/test phases.

In [ ]:
# ── 10.1  Metrics summary table ───────────────────────────────────────────────
checkpoint_exists = os.path.exists(CHECKPOINT_PATH)
train_ran         = len(train_losses) > 0
test_ran          = args.do_test and checkpoint_exists

print('=' * 60)
print('        GraphCodeBERT Clone Detection — Results Summary')
print('=' * 60)

if train_ran:
    final_loss = train_losses[-1][1]
    best_eval_f1 = max(eval_f1_scores) if eval_f1_scores else float('nan')
    print(f'  Training')
    print(f'    Epochs trained      : {args.epochs}')
    print(f'    Total steps         : {train_losses[-1][0]}')
    print(f'    Final training loss : {final_loss:.4f}')
    print(f'    Best val F1 (ckpt)  : {best_eval_f1:.4f}')
    print()

if checkpoint_exists and args.do_eval:
    print(f'  Validation (best checkpoint)')
    print(f'    Threshold           : {best_threshold:.2f}')
    print(f'    Accuracy            : {acc:.4f}')
    print(f'    Precision           : {results["eval_precision"]:.4f}')
    print(f'    Recall              : {results["eval_recall"]:.4f}')
    print(f'    F1                  : {results["eval_f1"]:.4f}')
    print(f'    ROC AUC             : {roc_auc:.4f}')
    print(f'    Average Precision   : {avg_prec:.4f}')
    print()

if test_ran:
    print(f'  Test set (threshold={best_threshold:.2f})')
    print(f'    Accuracy            : {test_acc:.4f}')
    print(f'    Precision           : {test_p:.4f}')
    print(f'    Recall              : {test_r:.4f}')
    print(f'    F1                  : {test_f1:.4f}')

print('=' * 60)

In [ ]:
# ── 10.2  Dashboard visualisations ────────────────────────────────────────────
has_train_plots = train_ran and len(eval_steps) > 0
has_eval_plots  = checkpoint_exists and args.do_eval
has_test_plots  = test_ran

if not (has_train_plots or has_eval_plots or has_test_plots):
    print('Run training and/or evaluation cells first to see summary plots.')
else:
    fig = plt.figure(figsize=(18, 10))
    gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

    # ── Row 1, Col 1: Training loss ───────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    if train_ran:
        s, l = zip(*train_losses)
        ax1.plot(s, l, color='steelblue', linewidth=1)
        ax1.set_title('Training Loss', fontweight='bold')
        ax1.set_xlabel('Step')
        ax1.set_ylabel('Loss')
        ax1.grid(True, linestyle='--', alpha=0.4)
    else:
        ax1.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax1.transAxes)

    # ── Row 1, Col 2: Validation F1 over checkpoints ─────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    if has_train_plots:
        ax2.plot(eval_steps, eval_f1_scores,        'o-', color='steelblue',      label='F1')
        ax2.plot(eval_steps, eval_precision_scores,  's-', color='mediumseagreen', label='Precision')
        ax2.plot(eval_steps, eval_recall_scores,     '^-', color='darkorange',     label='Recall')
        ax2.set_title('Val Metrics at Checkpoints', fontweight='bold')
        ax2.set_xlabel('Step')
        ax2.set_ylabel('Score')
        ax2.set_ylim(0, 1.05)
        ax2.legend(fontsize=9)
        ax2.grid(True, linestyle='--', alpha=0.4)
    else:
        ax2.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax2.transAxes)

    # ── Row 1, Col 3: Learning rate ───────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    if train_ran:
        sl, lr_vals = zip(*learning_rates)
        ax3.plot(sl, lr_vals, color='darkorange', linewidth=1)
        ax3.set_title('Learning Rate Schedule', fontweight='bold')
        ax3.set_xlabel('Step')
        ax3.set_ylabel('LR')
        ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1e}'))
        ax3.grid(True, linestyle='--', alpha=0.4)
    else:
        ax3.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax3.transAxes)

    # ── Row 2, Col 1: Confusion matrix ───────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 0])
    if has_eval_plots:
        cm_small = confusion_matrix(all_labels, y_preds)
        sns.heatmap(cm_small, annot=True, fmt='d', cmap='Blues', ax=ax4,
                    xticklabels=['Non-Clone', 'Clone'],
                    yticklabels=['Non-Clone', 'Clone'])
        ax4.set_title('Confusion Matrix (Val)', fontweight='bold')
        ax4.set_xlabel('Predicted')
        ax4.set_ylabel('Actual')
    else:
        ax4.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax4.transAxes)

    # ── Row 2, Col 2: ROC curve ───────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[1, 1])
    if has_eval_plots:
        ax5.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC={roc_auc:.4f}')
        ax5.plot([0, 1], [0, 1], 'k--', linewidth=1)
        ax5.set_title('ROC Curve (Val)', fontweight='bold')
        ax5.set_xlabel('FPR')
        ax5.set_ylabel('TPR')
        ax5.legend(fontsize=9)
        ax5.grid(True, linestyle='--', alpha=0.4)
    else:
        ax5.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax5.transAxes)

    # ── Row 2, Col 3: Final metric bar chart ──────────────────────────────────
    ax6 = fig.add_subplot(gs[1, 2])
    metric_names  = []
    metric_values = []
    metric_colors = []
    if has_eval_plots:
        metric_names  += ['Val\nAccuracy', 'Val\nPrecision', 'Val\nRecall', 'Val\nF1']
        metric_values += [acc, results['eval_precision'], results['eval_recall'], results['eval_f1']]
        metric_colors += ['steelblue'] * 4
    if has_test_plots:
        metric_names  += ['Test\nAccuracy', 'Test\nPrecision', 'Test\nRecall', 'Test\nF1']
        metric_values += [test_acc, test_p, test_r, test_f1]
        metric_colors += ['darkorange'] * 4

    if metric_names:
        bars = ax6.bar(metric_names, metric_values, color=metric_colors, alpha=0.85)
        for bar, val in zip(bars, metric_values):
            ax6.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)
        ax6.set_ylim(0, 1.15)
        ax6.set_title('Final Metrics Summary', fontweight='bold')
        ax6.set_ylabel('Score')
        ax6.grid(axis='y', linestyle='--', alpha=0.4)

        val_patch  = mpatches.Patch(color='steelblue',  alpha=0.85, label='Validation')
        test_patch = mpatches.Patch(color='darkorange', alpha=0.85, label='Test')
        ax6.legend(handles=[val_patch, test_patch], fontsize=9)
    else:
        ax6.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax6.transAxes)

    fig.suptitle('GraphCodeBERT Clone Detection — Results Dashboard',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.show()

In [ ]:
# Quick stats dump
print(f"=== KEY METRICS ===")
print(f"Best threshold: {best_threshold:.2f}")
print(f"Val Accuracy:   {acc:.4f}")
print(f"Val Precision:  {results['eval_precision']:.4f}")
print(f"Val Recall:     {results['eval_recall']:.4f}")
print(f"Val F1:         {results['eval_f1']:.4f}")
print(f"ROC AUC:        {roc_auc:.4f}")
print(f"Avg Precision:  {avg_prec:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Precision: {test_p:.4f}")
print(f"Test Recall:    {test_r:.4f}")
print(f"Test F1:        {test_f1:.4f}")
print(f"Final train loss: {train_losses[-1][1]}")
print(f"Best eval F1 during training: {max(eval_f1_scores):.4f}")
print(f"\n=== Dataset Sizes ===")
print(f"Train: {len(train_pairs)}, Valid: {len(valid_pairs)}, Test: {len(test_pairs)}")
total = len(train_pairs)+len(valid_pairs)+len(test_pairs)
print(f"Ratios: {len(train_pairs)/total*100:.1f}% / {len(valid_pairs)/total*100:.1f}% / {len(test_pairs)/total*100:.1f}%")
print(f"\n=== Confusion Matrix (Val) ===")
cm_v = confusion_matrix(all_labels, y_preds)
tn, fp, fn, tp = cm_v.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Sensitivity (Recall/TPR): {tp/(tp+fn) if (tp+fn)>0 else 0:.4f}")
print(f"Specificity (TNR):        {tn/(tn+fp) if (tn+fp)>0 else 0:.4f}")
print(f"\n=== Confusion Matrix (Test) ===")
cm_t = confusion_matrix(test_labels_all, test_preds)
tn2, fp2, fn2, tp2 = cm_t.ravel()
print(f"TN={tn2}, FP={fp2}, FN={fn2}, TP={tp2}")
print(f"Sensitivity (Recall/TPR): {tp2/(tp2+fn2) if (tp2+fn2)>0 else 0:.4f}")
print(f"Specificity (TNR):        {tn2/(tn2+fp2) if (tn2+fp2)>0 else 0:.4f}")
print(f"\n=== Checkpoint Eval History ===")
for s, f1, p, r in zip(eval_steps, eval_f1_scores, eval_precision_scores, eval_recall_scores):
    print(f"  Step {s:5d}: F1={f1:.4f}  P={p:.4f}  R={r:.4f}")